# Hybrid coarse-FD + FNO residual surrogate — Colab (A100) run

End-to-end reproduction of the Schwarzschild hybrid surrogate on a Colab GPU, using the **exact deployed configuration** (`configs/hybrid_sw_gate_s1em3.yaml`, batch 8, 100 epochs). This notebook only *calls* the repository scripts — it does **not** modify any core code.

**Before running:** `Runtime ▸ Change runtime type ▸ A100 GPU` (Pro+). An L4 (24 GB) or V100 (16 GB) also work — only the 12 GB laptop GPU did not, because batch-8 activations at 501×1001 resolution do not fit in 12 GB.

Pipeline: **clone → install → build corpus → train → evaluate (incl. observer-position sweep) → download results.**

Reproduction targets from the paper: validation MSE floor ≈ `4.0e-7`, canonical RMSD ≈ `6.85e-4`, population field-MSE ratio ≈ `79×`.

## 1 · Confirm the GPU
Want an A100 (40 GB). If you see a smaller card, that is fine as long as it has ≥ 16 GB.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2 · Clone the public repository

In [ ]:
import os

REPO_URL = "https://github.com/ScreaM835/QNM_Project_Fresh.git"
REPO_DIR = "/content/QNM_Project_Fresh"

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git log -1 --oneline

## 3 · Install dependencies
`neuraloperator` is pinned to **2.0.0**: the hybrid model only builds correctly on this version (a newer release changes the FNO API). Colab already ships a compatible PyTorch, so we do not reinstall torch.

In [ ]:
!pip -q install "neuraloperator==2.0.0" "numpy>=1.24" "scipy>=1.10" "pyyaml>=6.0" "tqdm>=4.66"

# Allocator fix for 40 GB A100s (harmless on H100/80GB): set process-wide so
# every subsequent !command inherits it, whatever the training cell says.
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_CUDA_ALLOC_CONF =", os.environ["PYTORCH_CUDA_ALLOC_CONF"])

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 4 · Build the coarse/fine FD corpus
Regenerated on Colab (deterministic, seed 0) rather than uploaded — the build is CPU-only and takes a few minutes. Produces the `k=4` prior corpus and the `k=2` corpus used to construct the label-free Richardson target.

In [ ]:
!python scripts/build_hybrid_dataset.py --config configs/hybrid_sw_dataset.yaml --k 4 --out outputs/hybrid/dataset_sw_k4.npz
!python scripts/build_hybrid_dataset.py --config configs/hybrid_sw_dataset.yaml --k 2 --out outputs/hybrid/dataset_sw_k2.npz

## 5 · Train the hybrid surrogate
Exact deployed config: gate scale `s = 1e-3`, label-free Richardson target, batch 8, 100 Adam epochs, seed 1234. Writes `model_best.pt`, `model_last.pt`, and the training history to `outputs/hybrid/fno_sw_gate_s1em3/`.

`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True` is required on Colab's **40 GB** A100s: the original runs used 80 GB A100s (CSD3 Wilkes3) and the run peaks near 36 GB, so the allocator must be allowed to use fragmented reserve. The flag changes memory management only — it has **no effect on the numerical results**.

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python scripts/train_hybrid_fno.py --config configs/hybrid_sw_gate_s1em3.yaml

## 6 · Evaluate — field accuracy + QNM extraction
Runs the 100-BH test-set field metrics and the M1–M5 extraction. The `--xq` flag drives the **observer-position sweep**: here we extract at `x_q ∈ {2, 5, 10, 15, 20} M` (against the FD reference at the same columns) on the fixed `[10, 50] M` window, to test whether a more asymptotic observer is preferable to `x_q = 2 M` given that the time domain was not extended.

In [ ]:
!python scripts/eval_hybrid_sw.py --config configs/hybrid_sw_gate_s1em3.yaml --xq 2 5 10 15 20 --t_start 10 --t_end 50

## 7 · Package the results for download
Zips the trained model, history, evaluation reports and figures. Download the archive, unzip into `outputs/hybrid/fno_sw_gate_s1em3/` in your local workspace, and the report figures/tables regenerate as normal.

In [ ]:
import shutil, os

src = "outputs/hybrid/fno_sw_gate_s1em3"
archive = "/content/hybrid_sw_gate_s1em3_colab"
shutil.make_archive(archive, "zip", src)
print("created", archive + ".zip",
      f"({os.path.getsize(archive + '.zip')/1e6:.1f} MB)")

try:
    from google.colab import files
    files.download(archive + ".zip")
except Exception as e:
    print("Auto-download unavailable; find the zip in the file browser at", archive + ".zip", "|", e)

### Optional · Save straight to Google Drive
If you prefer Drive over a browser download, run this instead of (or after) cell 7.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import shutil
shutil.copy("/content/hybrid_sw_gate_s1em3_colab.zip",
            "/content/drive/MyDrive/hybrid_sw_gate_s1em3_colab.zip")
print("copied to Drive: MyDrive/hybrid_sw_gate_s1em3_colab.zip")